<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [14]:
import sys
import os
sys.path.append(os.path.abspath("../../.."))

from config.spark_config import SparkConfig
from utils.logger import LoggerFactory
from config.io_config import *
from app.platform_app import PlatformApp
from utils.data_quality import *
from utils.data_cleaning import *
from utils.utils import *
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Set up</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Set up
    </h1>
</div>


In [15]:
# Initialize shared logger (all logs in this run go to the same file: etl_<run_id>.log)
logger = LoggerFactory.setup_logger(name="ETL", log_dir=LOG_DIR)

# Create Spark session with logging enabled (for tracing Spark-related operations)
spark = SparkConfig.create_spark(app_name="Paypal Analytic", logger=logger, use_databricks=True)

# Initialize main application with Spark and logger (used across ETL pipeline)
app = PlatformApp(spark=spark, logger=logger, catalog_name="paypal_analytic")

2026-04-08 22:12:26 | INFO     | ETL | logger.py:113 | Logger initialized | level=DEBUG | file=C:/01_Data/05-data-engineer-bootcamp/03_paypal_databricks/logs\etl_20260401_193104_713773.log
2026-04-08 22:12:28 | INFO     | ETL | spark_config.py:89 | Connected to Databricks via Spark Connect.
2026-04-08 22:12:28 | INFO     | ETL | platform_app.py:44 | Initializing Data Platform...
2026-04-08 22:12:28 | INFO     | ETL | platform_app.py:50 | Spark session initialized


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Silver</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Silver
    </h1>
</div>


In [16]:
df_bronze_disputes = spark.sql(f"SELECT * FROM {BRONZE_DISPUTE_TRANSACTIONS}")

# Preview result
df_bronze_disputes.show(n=10, truncate=False)

+------------------+------------------------------+------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------+--------+------------------------

## Transformations

### Select Features

In [17]:
df_silver_disputes_outcome = df_bronze_disputes.select("dispute_id", "create_time", "update_time", "dispute_outcome", 
                                                            "elton_created_at", "dt", "hour")

# Preview result
df_silver_disputes_outcome.show(n=10, truncate=False)

+------------------+------------------------------+------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+--------+----+
|dispute_id        |create_time                   |update_time                   |dispute_outcome                                                                                                                                                  |elton_created_at              |dt      |hour|
+------------------+------------------------------+------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+--------+----+
|PP-R-IQQ-501148533|2023-10-21 22:33:17.000000 UTC|2023-10-31 22:51:43.000000 UTC|{"outcome_code": "RESOLVED_BUYER_FAVOUR", "outco

### Check NULL

In [18]:
# check null
check_null(df = df_silver_disputes_outcome, logger=logger)

2026-04-08 22:12:34 | WARNING  | ETL | data_quality.py:65 | 
+-----------------+---------------+-----------+
|    Features     | Missing_Count | Missing_% |
+-----------------+---------------+-----------+
| dispute_outcome |      154      |   60.63   |
+-----------------+---------------+-----------+
2026-04-08 22:12:34 | WARNING  | ETL | data_quality.py:70 | Total missing values: 154 out of 254 rows.


### Trim spaces

In [19]:
# Remove those trim values
df_silver_disputes_outcome = clean_dataframe(df=df_silver_disputes_outcome)

# Preview result
df_silver_disputes_outcome.show(n=10, truncate=False)

+------------------+------------------------------+------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+--------+----+
|dispute_id        |create_time                   |update_time                   |dispute_outcome                                                                                                                                                  |elton_created_at              |dt      |hour|
+------------------+------------------------------+------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+--------+----+
|PP-R-IQQ-501148533|2023-10-21 22:33:17.000000 UTC|2023-10-31 22:51:43.000000 UTC|{"outcome_code": "RESOLVED_BUYER_FAVOUR", "outco

### Duplicates

In [20]:
df_silver_disputes_outcome = dedup(
    df_silver_disputes_outcome,
    dedup_cols=["dispute_id"],
    order_cols=["update_time", "dt", "hour", "elton_created_at"],
    logger=logger
)

2026-04-08 22:12:35 | INFO     | ETL | utils.py:190 | Starting deduplication
2026-04-08 22:12:35 | INFO     | ETL | utils.py:191 | Dedup columns: ['dispute_id']
2026-04-08 22:12:35 | INFO     | ETL | utils.py:192 | Order columns: ['update_time', 'dt', 'hour', 'elton_created_at']
2026-04-08 22:12:35 | INFO     | ETL | utils.py:210 | Order direction (desc): [True, True, True, True]
2026-04-08 22:12:35 | INFO     | ETL | utils.py:211 | Nulls last: True
2026-04-08 22:12:36 | INFO     | ETL | utils.py:218 | Input row count: 254
2026-04-08 22:12:37 | INFO     | ETL | utils.py:253 | Output row count after dedup: 82
2026-04-08 22:12:37 | INFO     | ETL | utils.py:254 | Removed duplicate rows: 172
2026-04-08 22:12:37 | INFO     | ETL | utils.py:255 | Deduplication completed


### Extract Data

In [21]:
disputed_outcome_schema = StructType([
    StructField("outcome_code", StringType()),
    StructField("outcome_reason", StringType()),
    StructField("amount_refunded", StructType([
        StructField("currency_code", StringType()),
        StructField("value", StringType())
    ]))
])

# Parse JSON -> struct (avoid multiple parsing)
df = df_silver_disputes_outcome.withColumn(
    "outcome",
    F.from_json(F.col("dispute_outcome"), disputed_outcome_schema)
)

# Preview result
df.show(n=10, truncate=False)

+------------------+------------------------------+------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+--------+----+------------------------------------------------------------------+
|dispute_id        |create_time                   |update_time                   |dispute_outcome                                                                                                                                   |elton_created_at              |dt      |hour|outcome                                                           |
+------------------+------------------------------+------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+--------+----+------------------------------------------

In [22]:
df_silver_disputes_outcome_final = df.select(
    "dispute_id",

    # Standardize timestamp
    parse_timestamp(F.col("create_time")).alias("create_time"),
    parse_timestamp(F.col("update_time")).alias("update_time"),

    # outcome_code: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("outcome.outcome_code")) == "", None)
         .otherwise(F.trim(F.col("outcome.outcome_code"))),
        F.lit("Unknown")
    ).alias("outcome_code"),

    # outcome_reason: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("outcome.outcome_reason")) == "", None)
         .otherwise(F.trim(F.col("outcome.outcome_reason"))),
        F.lit("Unknown")
    ).alias("outcome_reason"),

    # gross_amount: cast to decimal
    F.col("outcome.amount_refunded.value").cast("decimal(18,2)").alias("amount_refunded"),

    # outcome_reason: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("outcome.amount_refunded.currency_code")) == "", None)
         .otherwise(F.trim(F.col("outcome.amount_refunded.currency_code"))),
        F.lit("USD")
    ).alias("currency_code"),

    # Metadata timestamps
    parse_timestamp(F.col("elton_created_at")).alias("elton_created_at"),
    parse_timestamp2(F.col("dt")).cast("date").alias("dt"),
    F.col("hour").cast("int").alias("hour")
) \
.filter(F.col("dispute_id").isNotNull()) \
.withColumn("process_timestamp", F.date_trunc("second", F.current_timestamp()))

# Preview result
df_silver_disputes_outcome_final.show(n=10, truncate=False)

+------------------+-------------------+-------------------+----------------------+----------------------------------+---------------+-------------+-------------------+----------+----+-------------------+
|dispute_id        |create_time        |update_time        |outcome_code          |outcome_reason                    |amount_refunded|currency_code|elton_created_at   |dt        |hour|process_timestamp  |
+------------------+-------------------+-------------------+----------------------+----------------------------------+---------------+-------------+-------------------+----------+----+-------------------+
|PP-R-AEH-502867238|2023-11-05 22:23:40|2023-11-25 22:31:51|RESOLVED_SELLER_FAVOUR|INELIGIBLE_BUYER_PROTECTION_POLICY|NULL           |USD          |2024-03-07 04:04:26|2024-03-07|4   |2026-04-08 15:12:38|
|PP-R-AXM-497286450|2023-09-18 21:12:45|2023-09-28 21:25:24|RESOLVED_BUYER_FAVOUR |NO_SELLER_RESPONSE                |175.00         |USD          |2024-03-07 04:04:26|2024-03-07|4

### Transformed data to Silver Layer

In [23]:
if not spark.catalog.tableExists(SILVER_PATH_DISPUTED_PP02_DISPUTES_OUTCOME):
    logger.info("Silver disputed pp02 disputes outcome table not found. Creating new table...")
    df_silver_disputes_outcome_final.write.format("delta") \
                   .option("delta.enableChangeDataFeed", "true") \
                   .option("mergeSchema", "true") \
                   .mode("append") \
                   .saveAsTable(SILVER_PATH_DISPUTED_PP02_DISPUTES_OUTCOME)
    logger.info("Silver disputed pp02 disputes outcome table created successfully")
else:
    logger.info("Silver disputed pp02 disputes outcome table exists. Performing upsert...")
    upsert(spark=spark, df=df_silver_disputes_outcome_final, key_cols=["dispute_id"],
           table=SILVER_TABLE_DISPUTED_PP02_DISPUTES_OUTCOME, cdc="update_time",
           name_catalog=app.catalog_name, name_schema=SCHEMA_SILVER, logger=logger)
    logger.info("Upsert completed successfully")

2026-04-08 22:12:40 | INFO     | ETL | 546589265.py:10 | Silver disputed pp02 disputes outcome table exists. Performing upsert...
2026-04-08 22:12:40 | INFO     | ETL | utils.py:349 | Starting UPSERT into paypal_analytic.silver.disputed_pp02_disputed_outcome
2026-04-08 22:12:50 | INFO     | ETL | utils.py:379 | UPSERT completed successfully: paypal_analytic.silver.disputed_pp02_disputed_outcome
2026-04-08 22:12:50 | INFO     | ETL | 546589265.py:14 | Upsert completed successfully


In [24]:
app.stop()

2026-04-08 22:12:50 | INFO     | ETL | platform_app.py:259 | Stopping Spark session...
2026-04-08 22:12:51 | INFO     | ETL | platform_app.py:261 | Spark stopped.
